# M1 CPU backend on a cheap Colab CPU runtime

Runs the skeleton's CPU-backend configuration (Q4_0 on llama.cpp's CPU build, 20 tasks × 2 seeds = 40 episodes) and the CPU speed table (llama-bench pp512/tg128 for all five model files), then releases the runtime.

* Runtime: **CPU** (Runtime → Change runtime type). Costs a fraction of a compute unit per hour.
* Writes into the same experiment folder on Drive as the GPU notebook, so `report.md` combines both. Resumable.
* ~6–7 h on 2 cores (each step re-reads a ~6k-token prompt; the prompt cache is off by protocol). Development data.

In [ ]:
# 1. Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive'
PILOT_MODELS = f'{DRIVE}/CMPE295A_pilot_2026-10-08/models'
CACHE = f'{DRIVE}/CMPE295A_m1_cache'
OUT = f'{DRIVE}/CMPE295A_m1_runs'
BRANCH = 'm1/skeleton'
os.environ.update(TESTBED_OUT=OUT, TESTBED_MODELS='/content/models',
                  LLAMA_CPU_BIN='/content/cmpe_295a/llama.cpp/build-cpu/bin/llama-server')
THREADS = os.cpu_count()
!nproc; grep -m1 'model name' /proc/cpuinfo; free -g | head -2

In [ ]:
# 2. Code
import subprocess
if not os.path.exists('/content/cmpe_295a'):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/VritikaMalhotra/cmpe_295a', '/content/cmpe_295a'], check=True)
else:
    subprocess.run(['git', '-C', '/content/cmpe_295a', 'pull', '--ff-only'], check=True)
%cd /content/cmpe_295a
!git log -1 --oneline

In [ ]:
# 3. Environment + llama.cpp CPU build (restored from the Drive cache; llama-bench compiled here)
!bash scripts/setup_env.sh --llama --cache {CACHE}

In [ ]:
# 4. Model files: the five GPU-run files (pilot BF16/Q4_K_M/Q4_0 + Q8_0 + R3), SHA-256 recorded
!mkdir -p /content/models
!cp -n {PILOT_MODELS}/Qwen3-0.6B-BF16.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_K_M.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_0-pure.gguf /content/models/
!cp -n {CACHE}/Qwen3-0.6B-Q8_0.gguf {CACHE}/r3/Qwen3-0.6B-S7QAD-Q4_0-pure.gguf /content/models/
!cd /content/models && sha256sum *.gguf

In [ ]:
# 5. CPU speed table (C9): llama-bench pp512 / tg128, 3 repetitions, all five files
!bash scripts/bench.sh cpu /content/models {OUT}/bench {THREADS}

In [ ]:
# 6. The 40 CPU-backend episodes (re-run after a disconnect; it resumes)
!.venv/bin/python -m testbed.run +experiment=skeleton local=true backend=cpu cpu_threads={THREADS} 2>&1 | tee -a {OUT}/colab_cpu_run.log | grep -v '^  multi_turn' | tail -20
!.venv/bin/python -m testbed.run +experiment=skeleton report_only=true 2>&1 | tail -1

In [ ]:
# 7. Release this runtime (results are on Drive). Set RELEASE = False to keep it.
RELEASE = True
if RELEASE:
    from google.colab import runtime
    runtime.unassign()